# Show all the plots produced by DRP stage2 (recalibrate)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-12 : stage 2 ; (plots of the stage2 RUN collections, fgcm plots by theme and by cycle)
- purpose : display every plot (dataset types ending with `Plot`) **written by** `DRP.yaml#stage2-recalibrate` :
  - the `fgcm_Cycle<N>_<name>_Plot` plots of `fgcmFitCycle` (step 2a), about 40 per fit cycle
  - the `lsst.analysis.tools` plots of the analysis tasks (association, astrometric reference match, ...) when they exist
- helpers : `showplots_utils.py` (same folder), shared with the stage 1 notebook
- previous : `nb2026-11-11_ShowPlots_DRPstage1-single-visit.ipynb`
- see also : `nb2026-11-11_CheckCollections_DRPstage2-recalibrate.ipynb` (the datasets of stage 2, and the fgcm fitted parameters versus time)

The plots are PNG files written by the pipeline : they are retrieved from the Butler and displayed from disk (they cannot be read back with `butler.get`).

- the quantum graph for DRP.yaml#stage2-recalibrate: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage2-recalibrate.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import os
import re
import logging
import warnings
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from lsst.daf.butler import Butler

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import PlotBrowser, show_diagram

In [ ]:
# Turn the Butler.collections FutureWarning into an error (same convention as the CheckCollections notebooks)
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

butler = Butler('main')

## Configuration

In [ ]:
NB_TAG = 'ShowPlots_DRPstage2'

username = os.getenv('USER')
RUN_NAME = 'runelaiss1_20261111_tract2877patchsel_mediumpatcharray'

# Stage-2 output collection :
#   'stage2_v2' = output of bps_dp2cosmos_parsl_2026-11-03_stage2.yaml (whole stage2, fgcmFitCycle:minStarPerExp=100)
#   'stage2'    = output of bps_dp2cosmos_parsl_2026-11-03_step2a.yaml (step2a only : the fgcm tasks)
STAGE2_NAME = 'stage2_v2'

collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/{STAGE2_NAME}'

# Plot files retrieved from the Butler (a file already present is not downloaded again)
FIGDIR = Path(f'figs_{NB_TAG}_{STAGE2_NAME}')

# Quantum diagrams of the pipeline steps
DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')

# Safety limits when displaying many plots
MAX_TYPES = 6     # dataset types displayed per call (non-fgcm plots)
MAX_REFS = 12     # plots displayed per dataset type

# fgcm plots
FGCM_CYCLE = None       # fit cycle displayed (None = the last one found)
FGCM_MODE = 'single'    # 'single' : each plot at full resolution ; 'montage' : plots of one type side by side

print('stage2 collection :', collection_path)
print('figure directory  :', FIGDIR)

## RUN collections written by stage 2

The stage2 output collection is a CHAINED collection : the RUN collection(s) written by the stage2 submission(s), then the input collections (stage1, `LSSTCam/defaults`). Searching the whole chain would also return the **stage1** plots : only the RUN collections located under the stage2 collection name are searched, one plot per dataId (the latest run first).

In [ ]:
flat = list(butler.collections.query(collection_path, flatten_chains=True))
stage2_runs = [c for c in flat if c.startswith(collection_path + '/')]

print('stage2 collection :', collection_path)
print(f'{len(stage2_runs)} RUN collection(s) written by stage 2 :')
for run in stage2_runs:
    print('  ', run)

if not stage2_runs:
    raise RuntimeError(f'no RUN collection found under {collection_path} : check STAGE2_NAME')

## Quantum diagrams of the stage2 steps

In [ ]:
DIAGRAM_STEPS = {
    'stage2-recalibrate':           'DRP stage2 : recalibrate (whole stage)',
    'step2a-recalibrate-global':    'DRP step2a/global (fgcm)',
    'step2c-recalibrate-healpix':   'DRP step2c/healpix (gbdes)',
    'step2d-recalibrate-visits':    'DRP step2d/visits',
}

# Steps to display ; a missing file is reported and skipped
SHOW_STEPS = list(DIAGRAM_STEPS)

for step in SHOW_STEPS:
    show_diagram(DIAGRAM_DIR / f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png', title=DIAGRAM_STEPS[step])

## Plots in the stage2 collection

Only the dataset types matching `*Plot` are queried (much faster than listing all the datasets). The names are kept in full (`strip_prefix=False`) so that the regular expressions below are the same whatever the plots found.

In [ ]:
pb = PlotBrowser(butler, stage2_runs, raw_dir=FIGDIR, find_first=True, strip_prefix=False)

is_fgcm = pb.df['short'].str.startswith('fgcm_Cycle')
print('number of plots :', len(pb.df))
print('  fgcm          :', int(is_fgcm.sum()))
print('  other         :', int((~is_fgcm).sum()))

summary = pb.summary()
summary_fgcm = summary[summary.index.str.startswith('fgcm_Cycle')]
summary_other = summary[~summary.index.str.startswith('fgcm_Cycle')]

fgcm_cycles = []
if len(summary_fgcm):
    fg = pb.df[is_fgcm].copy()
    fg['cycle'] = fg['short'].str.extract('^fgcm_Cycle([0-9]+)_')[0].astype(int)
    fgcm_cycles = sorted(int(c) for c in fg['cycle'].unique())
    display(Markdown('**fgcm plots per fit cycle**'))
    display(fg.groupby('cycle').agg(dataset_types=('short', 'nunique'), plots=('short', 'size')))
    display(Markdown('**dimensions of the fgcm plot dataset types**'))
    display(summary_fgcm['dimensions'].value_counts().to_frame('dataset types'))

display(Markdown('**other plots (not fgcm)**'))
if len(summary_other):
    with pd.option_context('display.max_colwidth', None, 'display.width', 250, 'display.max_rows', 200):
        display(summary_other)
else:
    print('no plot other than fgcm : the analysis tasks of stage 2 have not written plots (yet)')

### How the other plots are split

Number of plots per dataset type and per value of each dimension (only when the number of values is small), for the plots that are not fgcm plots.

In [ ]:
for facet in ('band', 'visit', 'detector', 'tract'):
    table = pb.completeness(facet)
    if table.empty:
        continue
    table = table[~table.index.str.startswith('fgcm_Cycle')]
    if table.empty:
        continue
    if table.shape[1] <= 12:
        display(Markdown(f'**per {facet}**'))
        display(table)
    else:
        print(f'per {facet} : {table.shape[1]} different values (table not shown)')

## fgcm plots (step 2a, `fgcmFitCycle`)

`fgcmFitCycle` writes the plots `fgcm_Cycle<N>_<name>_Plot` at each fit cycle. They are grouped by theme below ; the cycle displayed is `FGCM_CYCLE` (default : the last one found). The last cell of this section compares one plot across all the cycles, to follow the convergence of the fit.

The themes are a reading guide made from the names of the plots ; the definition of each quantity is in the fgcm documentation.

In [ ]:
# Names of the plots (without the fgcm_Cycle<N>_ prefix and the _Plot suffix), by theme
FGCM_THEMES = {
    'zeropoints_throughput': ['Zeropoints', 'AbsThroughputs', 'FilterOffsets', 'I1', 'R1', 'I1R1', 'R1mI1',
                              'R1mI1Matchscale', 'R1mI1_vs_mjd'],
    'atmosphere':            ['NightlyAlpha', 'NightlyO3', 'NightlyPwv', 'NightlyTau', 'ModelPwvVsRpwv',
                              'RpwvVsRpwvInput', 'RpwvVsRpwvSmooth'],
    'exposure_gray':         ['Expgray', 'ExpgrayAirmass', 'ExpgrayCompareBands', 'ExpgrayCompareMjdRedblue',
                              'ExpgrayReference', 'ExpgrayUT', 'CompareRedblueExpgray'],
    'epsilon':               ['EpsilonGlobal', 'EpsilonDetector', 'EpsilonDetectorMatchscale'],
    'sigma':                 ['SigmaCal', 'SigmaRef', 'SigmaFgcmAllStars', 'SigmaFgcmPullsAllStars',
                              'SigmaFgcmReservedStars', 'SigmaFgcmReservedStarsCrunched',
                              'SigmaFgcmReservedStarsCrunchedNonPhotom', 'SigmaFgcmPullsReservedStars',
                              'SigmaFgcmPullsReservedStarsCrunched', 'SigmaFgcmPullsReservedStarsCrunchedNonPhotom'],
    'star_density':          ['DensityAllStarMap', 'DensityGoodStarMap'],
    'reference_residuals':   ['RefResidVsColorAll', 'RefResidVsColorCut'],
}

cycle = FGCM_CYCLE if FGCM_CYCLE is not None else (fgcm_cycles[-1] if fgcm_cycles else None)
print('fgcm cycles found :', fgcm_cycles, '| cycle displayed :', cycle)


def fgcm_pattern(names, cycle):
    '''Regex selecting the fgcm plot dataset types of the given names for one fit cycle.'''
    return '^fgcm_Cycle' + str(cycle) + '_(?:' + '|'.join(names) + ')_Plot$'


def show_fgcm_theme(theme):
    '''Display the plots of one theme for the selected cycle.'''
    if cycle is None:
        print('no fgcm plot available')
        return
    pb.show(fgcm_pattern(FGCM_THEMES[theme], cycle), mode=FGCM_MODE, max_refs=MAX_REFS)


def show_fgcm_across_cycles(name):
    '''Display the same plot (e.g. NightlyPwv) for all the fit cycles.'''
    pb.show('^fgcm_Cycle[0-9]+_' + name + '_Plot$', mode=FGCM_MODE, max_refs=MAX_REFS)


# Plots of the selected cycle that belong to no theme (new plots in a newer release, ...)
if cycle is not None:
    prefix = f'fgcm_Cycle{cycle}_'
    known = {n for names in FGCM_THEMES.values() for n in names}
    cycle_names = [s[len(prefix):-len('_Plot')] for s in summary_fgcm.index if s.startswith(prefix)]
    print(f'{len(cycle_names)} plot dataset types in cycle {cycle}')
    print('without theme :', sorted(set(cycle_names) - known))
    print('in a theme but not found :', sorted(known - set(cycle_names)))

### Zero points and throughput

`Zeropoints` : fgcm zero points ; `AbsThroughputs`, `FilterOffsets` : absolute throughputs and filter offsets ; `I1`, `R1`, `I1R1`, `R1mI1*` : the integrals of the instrumental passbands used by fgcm (`R1mI1_vs_mjd` : their difference versus time).

In [ ]:
show_fgcm_theme('zeropoints_throughput')

### Atmosphere

Nightly aerosol index (`NightlyAlpha`), ozone (`NightlyO3`), precipitable water vapour (`NightlyPwv`) and aerosol optical depth (`NightlyTau`), and the comparison of the fitted PWV with the retrieved one (`*Rpwv*`).

In [ ]:
show_fgcm_theme('atmosphere')

### Exposure gray (clouds, residual grey extinction)

`Expgray*` : the grey (achromatic) per-exposure correction, versus airmass, UT, MJD, for the different bands and for red / blue stars.

In [ ]:
show_fgcm_theme('exposure_gray')

### Epsilon (aperture / flat-field corrections)

`EpsilonGlobal`, `EpsilonDetector`, `EpsilonDetectorMatchscale` : the fitted epsilon (sky-background / aperture-correction) terms, global and per detector.

In [ ]:
show_fgcm_theme('epsilon')

### Sigma : repeatability and pulls

`SigmaFgcm*` : the repeatability of the fgcm magnitudes (all stars, reserved stars, crunched = after the fit, `NonPhotom` = non-photometric exposures), `SigmaFgcmPulls*` : the normalised residuals, `SigmaCal`, `SigmaRef` : the calibration and reference-star dispersions.

In [ ]:
show_fgcm_theme('sigma')

### Star density maps and reference residuals

`DensityAllStarMap`, `DensityGoodStarMap` : sky density of all the stars and of the good stars used by the fit ; `RefResidVsColor*` : residuals of the fgcm magnitudes with respect to the reference catalog, versus colour.

In [ ]:
show_fgcm_theme('star_density')
show_fgcm_theme('reference_residuals')

### The same plot for all the fit cycles

Change `NAME` to follow how a quantity evolves from cycle to cycle (e.g. `SigmaFgcmAllStars`, `NightlyPwv`, `Zeropoints`, `ExpgrayAirmass`). Only the cycles that wrote this plot are displayed.

In [ ]:
NAME = 'SigmaFgcmAllStars'

show_fgcm_across_cycles(NAME)

## Other plots : `lsst.analysis.tools` plots of stage 2

The plots that are not fgcm plots (stellar association repeatability, astrometric reference match, ...), grouped by kind of plot :

- `HistPlot` : distribution of a metric (repeatability, residuals, ...)
- `FocalPlanePlot` : a quantity binned in focal-plane coordinates (detector- or position-dependent effects)
- `WholeSkyPlot` : a quantity on the sky

One montage per dataset type, one panel per band (or per visit, ...) when the plots are split along that dimension. Set `KINDS_TO_SHOW` to a list such as `['HistPlot']` to restrict the display ; the limits `MAX_TYPES` / `MAX_REFS` of the configuration cell avoid flooding the notebook. These plots are written by the analysis tasks of the last steps of the stage (`updateVisitSummary` and the tasks after it) : there are none until they have run.

In [ ]:
KINDS_TO_SHOW = None   # None = all the kinds found, or e.g. ['HistPlot', 'FocalPlanePlot']

kinds = sorted(summary_other['kind'].unique()) if len(summary_other) else []
if KINDS_TO_SHOW is not None:
    kinds = [k for k in kinds if k in KINDS_TO_SHOW]
if not kinds:
    print('no plot other than fgcm to display')

for kind in kinds:
    display(Markdown(f'### {kind or "other plots"}'))
    pb.show('^(?!fgcm_).*' + kind + '$', max_types=MAX_TYPES, max_refs=MAX_REFS)

## Browse one dataset type

Pick a dataset type by its name in the summary tables above (fgcm or not) and show it as a montage or plot by plot at full resolution. Add `band='i'` (or `visit=...`, `tract=...`) to restrict the selection.

In [ ]:
TYPE_NAME = f'fgcm_Cycle{cycle}_Zeropoints_Plot' if cycle is not None else None

if TYPE_NAME is not None:
    print(TYPE_NAME)
    pb.show('^' + re.escape(TYPE_NAME) + '$', mode='single', max_refs=MAX_REFS)
    # side by side, one panel per band :
    # pb.show('^' + re.escape(TYPE_NAME) + '$', max_refs=MAX_REFS)

## Beyond the plots : the metrics behind them

List the metric datasets written by stage 2, and read one.

In [ ]:
metric_types = pb.sibling_types('metric')
print(f'{len(metric_types)} metric dataset type(s) in the stage2 run collection(s) :')
for name in metric_types:
    print('  ', name)

In [ ]:
# Pick one of the types listed above
METRIC_TYPE = metric_types[0] if metric_types else None

if METRIC_TYPE is not None:
    metrics = pb.load_first(METRIC_TYPE)
    if hasattr(metrics, 'to_pandas'):
        display(metrics.to_pandas().head())
    else:
        print(str(metrics)[:2000])